# Model Comparison: ARIMA vs ETS vs Prophet vs XGBoost vs Ensemble

Rolling-window backtest comparing 5 forecasting methods on monthly bookings ARR.

- **Training:** 3 years (36 months)
- **Forecast:** 12 months ahead (skip 1 month)
- **Roll:** advance 1 month until forecast end reaches Aug 2026
- **Metric:** WMAPE per horizon

In [ ]:
!pip install pmdarima prophet statsforecast lightgbm --quiet

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

## 1. Load Data

In [ ]:
query = """
WITH
fx AS (
    SELECT CURRENCY_CODE AS currency, TO_CHAR(STARTING_DATE, 'YYYY-MM') AS month_key, 1 / EXCHANGE_RATE AS rate
    FROM PROD_PROVISIONING.BUSINESS_CENTRAL.CURRENCY_EXCHANGE_RATE
),
nm(src, tgt) AS (
    SELECT * FROM VALUES
        ('Community Team','Community'),('APAC - Australia','Australia'),('APAC - Asia','Asia'),
        ('Retail - Enterprise','Retailer - Enterprise'),('Retail - Mid Market','Retailer - Mid Market'),
        ('SS - Analytics Ent','SS - Analytics Enterprise'),('SS - Analytics Mid','SS - Analytics Mid Market')
    AS t(src, tgt)
)
SELECT
    d.MONTH_CLOSED,
    ROUND(SUM(
        (CASE WHEN d.CURRENCY_CODE IN ('AUD','CAD','EUR')
              THEN d.BOOKINGS_ANNUALIZED_RECURRING_REVENUE / f.rate
              ELSE d.BOOKINGS_ANNUALIZED_RECURRING_REVENUE END)
        * os.SPLIT_PERCENTAGE / 100
    )) AS ARR_USD
FROM PROD_PROVISIONING.FINANCE_SHARE.DEALS_DATA d
JOIN PROD_PROVISIONING.SALESFORCE.OPPORTUNITY_SPLIT os
    ON os.OPPORTUNITY_ID = d.OPP_ID AND os.OPPORTUNITY_SPLIT_TYPE = 'Commissionable ARR'
JOIN PROD_PROVISIONING.SALESFORCE.OPPORTUNITY o
    ON o.OPPORTUNITY_ID = d.OPP_ID
LEFT JOIN fx f ON f.currency = d.CURRENCY_CODE AND f.mon = SUBSTR(d.MONTH_CLOSED, 6, 2)
LEFT JOIN nm ON nm.src = os.EMPLOYEE_REPORTING_GROUP
WHERE d.DEAL_STAGE = 'Closed Won'
    AND d.MONTH_CLOSED >= '2020-01' AND d.MONTH_CLOSED <= '2026-08'
    AND COALESCE(o.OPPORTUNITY_TYPE, '') <> 'Rate Reduction'
    AND COALESCE(d.BOOKING, '') <> 'Other'
    AND NOT (COALESCE(d.BOOKING, '') = '' AND d.BOOKINGS_ANNUALIZED_RECURRING_REVENUE < 0)
    AND os.EMPLOYEE_REPORTING_GROUP NOT IN
        ('Do Not Report', 'System Admin', 'Customer Success', 'Revenue Recovery')
GROUP BY 1
ORDER BY 1
"""

df = session.sql(query).to_pandas()
df['MONTH_CLOSED'] = df['MONTH_CLOSED'].astype(str)
print(f"Loaded {len(df)} months: {df['MONTH_CLOSED'].iloc[0]} to {df['MONTH_CLOSED'].iloc[-1]}")
df.head()

## 2. Define Forecasting Models

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import pmdarima as pm
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet
import lightgbm as lgb

# --- Model 1: auto_arima ---
def forecast_arima(train_vals, n_periods):
    model = pm.auto_arima(train_vals, seasonal=True, m=12, suppress_warnings=True,
                          stepwise=True, error_action='ignore',
                          max_p=3, max_q=3, max_P=2, max_Q=2, max_d=2, max_D=1)
    fc = model.predict(n_periods=n_periods)
    params = f"({model.order[0]},{model.order[1]},{model.order[2]})({model.seasonal_order[0]},{model.seasonal_order[1]},{model.seasonal_order[2]})[12]"
    return np.maximum(fc, 0), params

# --- Model 2: ETS (Holt-Winters) ---
def forecast_ets(train_vals, n_periods):
    try:
        model = ExponentialSmoothing(train_vals, seasonal_periods=12,
                                     trend='add', seasonal='mul',
                                     initialization_method='estimated').fit(optimized=True)
        fc = model.forecast(n_periods)
        params = f"trend={model.params.get('smoothing_trend',0):.2f},season={model.params.get('smoothing_seasonal',0):.2f}"
    except:
        model = ExponentialSmoothing(train_vals, seasonal_periods=12,
                                     trend='add', seasonal='add',
                                     initialization_method='estimated').fit(optimized=True)
        fc = model.forecast(n_periods)
        params = "additive_fallback"
    return np.maximum(fc, 0), params

# --- Model 3: Prophet ---
def forecast_prophet(train_dates, train_vals, n_periods):
    pdf = pd.DataFrame({'ds': train_dates, 'y': train_vals})
    model = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                    daily_seasonality=False, seasonality_mode='multiplicative')
    model.fit(pdf)
    future = model.make_future_dataframe(periods=n_periods, freq='MS')
    pred = model.predict(future)
    fc = pred['yhat'].iloc[-n_periods:].values
    return np.maximum(fc, 0), "prophet_mult"

# --- Model 4: XGBoost with lag features ---
def forecast_xgb(train_vals, n_periods):
    # Build lag features: t-1, t-2, t-3, t-6, t-12 + month_of_year
    def make_features(series):
        df = pd.DataFrame({'y': series})
        for lag in [1, 2, 3, 6, 12]:
            df[f'lag_{lag}'] = df['y'].shift(lag)
        df['month'] = [(i % 12) + 1 for i in range(len(df))]
        df['quarter_end'] = df['month'].isin([3, 6, 9, 12]).astype(int)
        return df.dropna()

    feat_df = make_features(train_vals)
    X = feat_df.drop('y', axis=1)
    y_train = feat_df['y']

    model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1, max_depth=4,
                               num_leaves=15, verbose=-1, random_state=42)
    model.fit(X, y_train)

    # Recursive forecast
    history = list(train_vals)
    preds = []
    for step in range(n_periods):
        h = history
        row = {}
        for lag in [1, 2, 3, 6, 12]:
            row[f'lag_{lag}'] = h[-lag] if lag <= len(h) else 0
        next_month = ((len(train_vals) + step) % 12) + 1
        row['month'] = next_month
        row['quarter_end'] = int(next_month in [3, 6, 9, 12])
        pred = model.predict(pd.DataFrame([row]))[0]
        pred = max(pred, 0)
        preds.append(pred)
        history.append(pred)

    return np.array(preds), "lgbm_lags"

print("All model functions defined.")

## 3. Rolling Backtest

In [ ]:
series = df.set_index('MONTH_CLOSED')['ARR_USD'].astype(float)
series.index = pd.PeriodIndex(series.index, freq='M')
series = series.sort_index()

TRAIN_MONTHS = 36
FORECAST_MONTHS = 12
SKIP = 1
last_period = pd.Period('2026-08', freq='M')

all_results = []

for i in range(len(series)):
    train_end_idx = i + TRAIN_MONTHS - 1
    if train_end_idx >= len(series):
        break

    train = series.iloc[i:train_end_idx + 1]
    train_end = train.index[-1]
    forecast_start = train_end + 2
    forecast_end = forecast_start + FORECAST_MONTHS - 1

    if forecast_end > last_period:
        break

    forecast_periods = pd.period_range(forecast_start, forecast_end, freq='M')
    actuals = series.reindex(forecast_periods)
    if actuals.isna().any():
        break

    train_vals = train.values
    train_dates = [p.to_timestamp() for p in train.index]
    n_total = SKIP + FORECAST_MONTHS

    # Run all models
    models = {}

    # ARIMA
    try:
        fc, params = forecast_arima(train_vals, n_total)
        models['arima'] = {'preds': fc[SKIP:], 'params': params}
    except Exception as e:
        models['arima'] = {'preds': np.full(FORECAST_MONTHS, np.nan), 'params': f'error: {e}'}

    # ETS
    try:
        fc, params = forecast_ets(train_vals, n_total)
        models['ets'] = {'preds': fc[SKIP:], 'params': params}
    except Exception as e:
        models['ets'] = {'preds': np.full(FORECAST_MONTHS, np.nan), 'params': f'error: {e}'}

    # Prophet
    try:
        fc, params = forecast_prophet(train_dates, train_vals, n_total)
        models['prophet'] = {'preds': fc[SKIP:], 'params': params}
    except Exception as e:
        models['prophet'] = {'preds': np.full(FORECAST_MONTHS, np.nan), 'params': f'error: {e}'}

    # XGBoost
    try:
        fc, params = forecast_xgb(train_vals, n_total)
        models['xgboost'] = {'preds': fc[SKIP:], 'params': params}
    except Exception as e:
        models['xgboost'] = {'preds': np.full(FORECAST_MONTHS, np.nan), 'params': f'error: {e}'}

    # Ensemble (mean of available models)
    valid_preds = [m['preds'] for m in models.values() if not np.isnan(m['preds']).any()]
    if valid_preds:
        ensemble_preds = np.mean(valid_preds, axis=0)
        models['ensemble'] = {'preds': ensemble_preds, 'params': f'avg_of_{len(valid_preds)}'}

    # Score each model
    for model_name, m in models.items():
        preds = m['preds']
        if np.isnan(preds).any():
            continue
        for h_idx in range(FORECAST_MONTHS):
            all_results.append({
                'train_end': str(train_end),
                'forecast_start': str(forecast_start),
                'forecast_end': str(forecast_end),
                'model': model_name,
                'horizon': h_idx + 1,
                'period': str(forecast_periods[h_idx]),
                'predicted': preds[h_idx],
                'actual': actuals.values[h_idx],
                'abs_error': abs(preds[h_idx] - actuals.values[h_idx]),
                'pct_error': abs(preds[h_idx] - actuals.values[h_idx]) / actuals.values[h_idx] * 100,
                'params': m['params']
            })

    window_num = len(set(r['train_end'] for r in all_results))
    if window_num % 5 == 0:
        print(f"  Window {window_num} complete: train through {train_end}")

print(f"\nDone. {window_num} windows, {len(all_results)} total scored cells.")

## 4. Results: WMAPE by Model × Horizon

In [ ]:
results_df = pd.DataFrame(all_results)

# WMAPE by model × horizon
wmape = results_df.groupby(['model', 'horizon']).apply(
    lambda g: pd.Series({
        'wmape': g['abs_error'].sum() / g['actual'].sum() * 100,
        'bias': (g['predicted'].sum() / g['actual'].sum() - 1) * 100,
        'n_windows': g['train_end'].nunique()
    })
).reset_index()

# Pivot for display
pivot = wmape.pivot_table(index='horizon', columns='model', values='wmape').round(1)
pivot = pivot[['arima', 'ets', 'prophet', 'xgboost', 'ensemble']]
pivot.columns = ['ARIMA', 'ETS', 'Prophet', 'XGBoost', 'Ensemble']

print("WMAPE (%) by Horizon × Model\n")
print(pivot.to_string())

# Best model per horizon
print("\n\n--- Best Model per Horizon ---")
for h in pivot.index:
    row = pivot.loc[h]
    best = row.idxmin()
    print(f"  h={h:2d}: {best:10s} ({row[best]:.1f}%)  |  " + "  ".join(f"{m}={v:.1f}" for m, v in row.items()))

## 5. Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. WMAPE by horizon (line chart)
ax = axes[0, 0]
for model in ['ARIMA', 'ETS', 'Prophet', 'XGBoost', 'Ensemble']:
    ax.plot(pivot.index, pivot[model], marker='o', label=model)
ax.set_xlabel('Horizon (months)')
ax.set_ylabel('WMAPE (%)')
ax.set_title('WMAPE by Forecast Horizon')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Average WMAPE across all horizons (bar chart)
ax = axes[0, 1]
avg_wmape = pivot.mean()
colors = ['steelblue', 'coral', 'seagreen', 'orchid', 'gold']
ax.bar(avg_wmape.index, avg_wmape.values, color=colors)
ax.set_ylabel('Avg WMAPE (%)')
ax.set_title('Average WMAPE Across All Horizons')
for i, v in enumerate(avg_wmape.values):
    ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)

# 3. WMAPE for short-term (h1-h3)
ax = axes[1, 0]
short = pivot.loc[1:3].mean()
ax.bar(short.index, short.values, color=colors)
ax.set_ylabel('Avg WMAPE (%)')
ax.set_title('Short-Term (h1-h3) Average WMAPE')
for i, v in enumerate(short.values):
    ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)

# 4. Win count by horizon
ax = axes[1, 1]
wins = pivot.idxmin(axis=1).value_counts()
ax.bar(wins.index, wins.values, color=[colors[list(pivot.columns).index(m)] for m in wins.index])
ax.set_ylabel('Horizons Won')
ax.set_title('Best Model by Horizon (Win Count)')

plt.tight_layout()
plt.show()

## 6. Save Results

In [ ]:
# Save detailed results
save_df = results_df[['train_end','model','horizon','period','predicted','actual','abs_error','pct_error','params']].copy()
save_df.columns = [c.upper() for c in save_df.columns]
snow_df = session.create_dataframe(save_df)
snow_df.write.mode('overwrite').save_as_table('PROD_PROVISIONING.FINANCE_SHARE.MODEL_COMPARISON_RESULTS')

# Save summary pivot
summary = wmape.copy()
summary.columns = [c.upper() for c in summary.columns]
snow_summary = session.create_dataframe(summary)
snow_summary.write.mode('overwrite').save_as_table('PROD_PROVISIONING.FINANCE_SHARE.MODEL_COMPARISON_SUMMARY')

print('Saved to PROD_PROVISIONING.FINANCE_SHARE.MODEL_COMPARISON_RESULTS')
print('Saved to PROD_PROVISIONING.FINANCE_SHARE.MODEL_COMPARISON_SUMMARY')